In [1]:
import cv2
import numpy as np
import os

# POTHOLE DETECTION USING IMAGE PROCESSING

video_path = "vdo3.MOV"

# Output folders
os.makedirs("results/original", exist_ok=True)
os.makedirs("results/enhanced", exist_ok=True)
os.makedirs("results/mask", exist_ok=True)
os.makedirs("results/pothole_detected", exist_ok=True)

# Load video
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Cannot open video file")
    exit()

frame_no = 0
save_no = 0

POTHOLE_COLOR = (0, 0, 255)   # Red color in BGR

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_no += 1

    # Process every 15th frame only
    if frame_no % 15 != 0:
        continue

    # Resize frame
    frame = cv2.resize(frame, (800, 500))
    original = frame.copy()

    # REGION OF INTEREST - ROAD AREA ONLY

    roi_mask = np.zeros((500, 800), dtype=np.uint8)

    road_points = np.array([[
        (170, 230),
        (650, 230),
        (730, 500),
        (80, 500)
    ]], dtype=np.int32)

    cv2.fillPoly(roi_mask, road_points, 255)

    # IMAGE ENHANCEMENT

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Improve contrast using CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # Noise reduction using Gaussian Blur
    blur = cv2.GaussianBlur(enhanced, (7, 7), 0)

    # SEGMENTATION

    # Adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        61,
        15
    )

    # Morphological operations
    kernel = np.ones((3, 3), np.uint8)

    # Remove small noise
    morph = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

    # Fill small gaps
    morph = cv2.morphologyEx(morph, cv2.MORPH_CLOSE, kernel, iterations=2)

    # REMOVE GREEN REGIONS

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    green_mask = cv2.inRange(
        hsv,
        np.array([35, 40, 40]),
        np.array([85, 255, 255])
    )

    morph[green_mask > 0] = 0

    # Apply road ROI mask
    morph = cv2.bitwise_and(morph, morph, mask=roi_mask)

    # CONTOUR DETECTION

    contours, _ = cv2.findContours(
        morph,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    final = frame.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)

        # Pothole size filtering
        if area < 2500 or area > 7000:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        # Avoid side false detections
        if x < 60 or x + w > 740:
            continue

        # Avoid very large unwanted regions
        if w > 220 or h > 180:
            continue

        aspect_ratio = w / float(h)

        # Pothole condition: medium/large blob-like region
        if 0.5 <= aspect_ratio <= 2.0:
            cv2.drawContours(final, [cnt], -1, POTHOLE_COLOR, 2)

            cv2.rectangle(
                final,
                (x, y),
                (x + w, y + h),
                POTHOLE_COLOR,
                2
            )

            cv2.putText(
                final,
                "POTHOLE",
                (x, y - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                POTHOLE_COLOR,
                2
            )

    # SAVE OUTPUTS

    cv2.imwrite(f"results/original/original_{save_no}.jpg", original)
    cv2.imwrite(f"results/enhanced/enhanced_{save_no}.jpg", enhanced)
    cv2.imwrite(f"results/mask/mask_{save_no}.jpg", morph)
    cv2.imwrite(f"results/pothole_detected/pothole_{save_no}.jpg", final)

    save_no += 1

    cv2.imshow("Pothole Detection", final)

    if cv2.waitKey(500) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

print("Pothole Detection Completed Successfully!")
print("Total saved frames:", save_no)

Pothole Detection Completed Successfully!
Total saved frames: 33
